# Solar Filament Segmentation — submission from a Hugging Face checkpointSubmission-only. There is no training here: this notebook clones the pipelinecode at a pinned revision (needed to run `predict.py`), then pulls the *modelweights* from a Hugging Face model repo instead of training them in-session.Use this to iterate on submissions quickly — post-processing tweaks, TTA,threshold changes — without paying for a full training run each time. Trainwith `notebooks/kaggle_runner.ipynb` (which pushes its best checkpoint to thesame HF repo when given `--hf-repo-id`), then submit from here.Requirements: *Internet* enabled (Settings -> Internet). GPU is optional —inference is much lighter than training and will run on CPU if needed.

In [ ]:
# --- the only cell you normally edit -----------------------------------------
REPO_URL   = "https://github.com/ShreyPatel1311/solar-filament-segmentation.git"
REVISION   = "main"                    # branch, tag, or full commit SHA
HF_REPO_ID = "your-username/filament-unet-r34"   # the model repo train.py pushed to
HF_FILENAME = "best_model.pt"          # matches --hf-filename at train time, if changed
# -----------------------------------------------------------------------------

In [ ]:
# Kaggle Secrets -> HF_TOKEN (Add-ons -> Secrets in the notebook editor).
# Only needs read access if HF_REPO_ID is public; a private repo needs a
# token with read access to it either way.
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception as exc:
    print("No HF_TOKEN secret found (fine for a public repo):", exc)

In [ ]:
import os, subprocess, sys, shutil, pathlib

WORK_DIR = pathlib.Path("/kaggle/working")
os.chdir(WORK_DIR)  # stand outside REPO_DIR before touching it, so this cell
                     # is safe to rerun without a kernel restart

REPO_DIR = WORK_DIR / "repo"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REVISION], check=True)

commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                        check=True, capture_output=True, text=True).stdout.strip()
print("running commit", commit)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
os.environ["PYTHONPATH"] = str(REPO_DIR / "src")

In [ ]:
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

!pip install -q -r requirements-kaggle.txt
!pip install -q --no-deps -r requirements-kaggle-nodeps.txt

In [ ]:
import albucore, albumentations, cv2, huggingface_hub, pycocotools, segmentation_models_pytorch as smp, torch

import filseg
from filseg.data.transforms import val_transforms
from filseg.paths import resolve_paths

print("filseg        ", filseg.__version__)
print("torch         ", torch.__version__, "| cuda:", torch.cuda.is_available())
print("albumentations", albumentations.__version__, "| albucore", albucore.__version__)
print("smp           ", smp.__version__, "| huggingface_hub", huggingface_hub.__version__)

paths = resolve_paths()
print("test images   ", len(list(paths.test_images.glob("*.jpeg"))))

## Predict the test set from the Hub checkpoint

In [ ]:
args = ["--hf-repo-id", HF_REPO_ID, "--hf-filename", HF_FILENAME,
        "--out", "/kaggle/working/submission.csv"]
!python scripts/predict.py {" ".join(args)}

## Score locally against the held-out validation split

Reuses the checkpoint `predict.py` already downloaded above -- no second
Hub download. Reports **two different PQ numbers**: `pq` pools TP/FP/FN/IoU
across every val image before computing one score (dominated by images with
many instances); `mean_image_pq` scores each image independently, then
averages (every image counts equally, so one bad image can swing it hard).
The competition's rules don't state which of these their own scorer computes
-- if the leaderboard score sits much closer to one than the other, that's a
strong signal which, and worth knowing before chasing the gap as a modelling
problem.

In [ ]:
CHECKPOINT_PATH = f"/kaggle/working/checkpoints/{HF_FILENAME}"  # what predict.py just downloaded
!python scripts/evaluate.py --checkpoint {CHECKPOINT_PATH}

In [ ]:
import pandas as pd

submission = pd.read_csv("/kaggle/working/submission.csv")
print(submission.shape, "rows |",
     submission.filament_id.str.rsplit("_", n=1).str[0].nunique(), "images")
submission.head()